# Assignment 1

You only need to write one line of code for each question. When answering questions that ask you to identify or interpret something, the length of your response doesn’t matter. For example, if the answer is just ‘yes,’ ‘no,’ or a number, you can just give that answer without adding anything else.

We will go through comparable code and concepts in the live learning session. If you run into trouble, start by using the help `help()` function in Python, to get information about the datasets and function in question. The internet is also a great resource when coding (though note that **no outside searches are required by the assignment!**). If you do incorporate code from the internet, please cite the source within your code (providing a URL is sufficient).

Please bring questions that you cannot work out on your own to office hours, work periods or share with your peers on Slack. We will work with you through the issue.

### Classification using KNN

Let's set up our workspace and use the **Wine dataset** from `scikit-learn`. This dataset contains 178 wine samples with 13 chemical features, used to classify wines into different classes based on their origin.

The **response variable** is `class`, which indicates the type of wine. We'll use all of the chemical features to predict this response variable.

In [90]:
# Import standard libraries
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import recall_score, precision_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [39]:
from sklearn.datasets import load_wine

# Load the Wine dataset
wine_data = load_wine()

# Convert to DataFrame
wine_df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)

# Bind the 'class' (wine target) to the DataFrame
wine_df['class'] = wine_data.target

# Display the DataFrame
wine_df


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,class
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2


#### **Question 1:** 
#### Data inspection

Before fitting any model, it is essential to understand our data. **Use Python code** to answer the following questions about the **Wine dataset**:

_(i)_ How many observations (rows) does the dataset contain?

In [20]:
# 178 rows × 14 columns
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  class

_(ii)_ How many variables (columns) does the dataset contain?

In [36]:
# Since info clearly counts how many columns above and lists them I used this. 14 columns

number_of_columns = len(wine_df.columns)
number_of_columns


14

_(iii)_ What is the 'variable type' of the response variable `class` (e.g., 'integer', 'category', etc.)? What are the 'levels' (unique values) of the variable?

In [25]:
print(wine_df['class'].dtype)
print(wine_df['class'].unique())
#variable type is integer, the values are 0, 1, and 2.


int64
[0 1 2]



_(iv)_ How many predictor variables do we have (Hint: all variables other than `class`)? 

In [ ]:
# Your answer here
13

You can use `print()` and `describe()` to help answer these questions.

#### **Question 2:** 
#### Standardization and data-splitting

Next, we must preform 'pre-processing' or 'data munging', to prepare our data for classification/prediction. For KNN, there are three essential steps. A first essential step is to 'standardize' the predictor variables. We can achieve this using the scaler method, provided as follows:

In [47]:
# Select predictors (excluding the last column)
wine_predictors = wine_df.iloc[:, :-1]

# Standardize the predictors
scaler = StandardScaler()
wine_predictors_standardized = pd.DataFrame(scaler.fit_transform(wine_predictors), columns=wine_predictors.columns)

# Display the head of the standardized predictors
print(wine_predictors_standardized.head())

    alcohol  malic_acid       ash  alcalinity_of_ash  magnesium  \
0  1.518613   -0.562250  0.232053          -1.169593   1.913905   
1  0.246290   -0.499413 -0.827996          -2.490847   0.018145   
2  0.196879    0.021231  1.109334          -0.268738   0.088358   
3  1.691550   -0.346811  0.487926          -0.809251   0.930918   
4  0.295700    0.227694  1.840403           0.451946   1.281985   

   total_phenols  flavanoids  nonflavanoid_phenols  proanthocyanins  \
0       0.808997    1.034819             -0.659563         1.224884   
1       0.568648    0.733629             -0.820719        -0.544721   
2       0.808997    1.215533             -0.498407         2.135968   
3       2.491446    1.466525             -0.981875         1.032155   
4       0.808997    0.663351              0.226796         0.401404   

   color_intensity       hue  od280/od315_of_diluted_wines   proline  
0         0.251717  0.362177                      1.847920  1.013009  
1        -0.293321  0.406051

(i) Why is it important to standardize the predictor variables?

> Because the measure of different predictor variables will be more effectively comparable. The influence of different units is mitigated so that one predictor variable doesn't exert an excessive influence over the results. It also ensures you don't end up with super huge or tiny numbers (overflow/underflow).

(ii) Why did we elect not to standard our response variable `Class`?

It is our target variable so we want the results to be interpretable as 0,1,2. The predictor variables need to be compared to each other effectively for the model to work, but the response variable wouldn't make sense as it is literally a class and could be A, B, C instead - it is a nominal number (nomen means name in latin which is the only way to remember to call it  'nominal')

(iii) A second essential step is to set a random seed. Do so below (Hint: use the random.seed function). Why is setting a seed important? Is the particular seed value important? Why or why not?

Setting the seed is important for result reproducability, debugging, testing and sharing. When someone else uses the seed, the data will be "shuffled" or "folded" in the same order. The seed value is important because it determines which seed (set of random numbers) you used to get your results. If someone uses a different seed, you can still run your program but the samples you are pulling from will be divided or shuffled slightly differently. One would hope the person testing your model would get similar but not identical results. 

(iv) A third essential step is to split our standardized data into separate training and testing sets. We will split into 75% training and 25% testing. The provided code randomly partitions our data, and creates linked training sets for the predictors and response variables. 

Extend the code to create a non-overlapping test set for the predictors and response variables.

In [106]:
# Reset indices for both predictors and response variable
wine_predictors_standardized_train = wine_predictors_standardized_train.reset_index(drop=True)
wine_class_train = wine_class_train.reset_index(drop=True)

# Combine predictors and response variable for the training set
wine_train = pd.concat([wine_predictors_standardized_train, wine_class_train], axis=1)

# Reset indices for the test set as well
wine_predictors_standardized_test = wine_predictors_standardized_test.reset_index(drop=True)
wine_class_test = wine_class_test.reset_index(drop=True)

# Combine predictors and response variable for the testing set
wine_test = pd.concat([wine_predictors_standardized_test, wine_class_test], axis=1)



#### **Question 4:**
#### Model evaluation

Using the best value for `n_neighbors`, fit a KNN model on the training data and evaluate its performance on the test set using `accuracy_score`.

In [108]:
# Your code here...
#I googled what factors determine wine flavour and it's looks like tanins, acidity and sugar. 
# Let's see the range of values for total_phenols (bitterness), and malic_acid which is acidity. 
# You can see the range of values for these two variables is pretty large. 
range_total_phenols = wine_train['total_phenols'].max() - wine_train['total_phenols'].min()
range_malic_acid = wine_train['malic_acid'].max() - wine_train['malic_acid'].min()
print(f"Range of total phenols: {range_total_phenols}")
print(f"Range of malic acid: {range_malic_acid}")

Range of total phenols: 4.646761490030702
Range of malic acid: 4.380596150508345


In [111]:
np.random.seed(123)

wine_predictors_standardized_train, wine_predictors_standardized_test, wine_class_train, wine_class_test = train_test_split(
    wine_predictors_standardized, wine_df["class"], train_size=0.75, shuffle=True, stratify=wine_df["class"]
)

knn = KNeighborsClassifier(n_neighbors=10)
knn


KNeighborsClassifier(n_neighbors=10)

In [112]:
X = wine_train[["total_phenols", "malic_acid"]]
y = wine_train["class"]

print(X.head())
print(y.head())
print(X.shape, y.shape)


   total_phenols  malic_acid
0      -0.633101   -1.208567
1       0.808997   -0.562250
2       0.889114   -0.472483
3       1.289697   -0.544297
4       1.610163   -0.418624
0    1
1    0
2    0
3    0
4    0
Name: class, dtype: int64
(133, 2) (133,)


In [113]:

print(wine_train)


      alcohol  malic_acid       ash  alcalinity_of_ash  magnesium  \
0   -0.828391   -1.208567 -1.522511          -1.409821   2.545825   
1    1.518613   -0.562250  0.232053          -1.169593   1.913905   
2    0.777454   -0.472483  1.218995          -0.689137   0.860705   
3    2.160950   -0.544297  0.085839          -2.430790  -0.613775   
4    1.703902   -0.418624  0.049285          -2.250619   0.158572   
..        ...         ...       ...                ...        ...   
128 -1.532492    0.308483  2.023170           0.151661   0.228785   
129 -0.494869   -0.894385 -1.705278          -0.298767  -0.824415   
130  0.925685   -0.544297  0.158946          -1.049479  -0.754202   
131 -0.445459   -0.876432 -1.266637          -0.809251   0.018145   
132  0.135116   -1.190614 -2.436346          -1.349764  -1.526548   

     total_phenols  flavanoids  nonflavanoid_phenols  proanthocyanins  \
0        -0.633101   -0.179981             -0.095517         2.048364   
1         0.808997    1.0

In [115]:
print(X.shape, y.shape)

(133, 2) (133,)


In [121]:

wine_test["predicted"] = knn.predict(wine_test[["total_phenols", "malic_acid"]])
wine_test[["class", "predicted"]]

,class,predicted
0,1,0
1,1,1
2,1,1
3,1,0
4,1,0
5,0,0
6,1,0
7,1,1
8,0,0
9,2,0


In [126]:
knn.score(
    wine_train[["total_phenols", "malic_acid"]],
    wine_train["class"],
)

0.7819548872180451

In [140]:
pd.crosstab(
    wine_train["class"],
    wine_test["predicted"],
    rownames = ['Acutal'],
    colnames = ['Predicted']
)

Predicted,0,1,2
Acutal,,,
0,9,4,2
1,10,4,5
2,8,0,3


In [151]:

# Class 0: 55.6% precision, Class 1: 87.5% precision, Class 2: 90% precision. 
# Guess the model is bad at predicting class 0, pretty good at predicting class 1 and 2.


precision_score(
    y_true=wine_test["class"],
    y_pred=wine_test["predicted"],
    average=None  
)


array([0.55555556, 0.875     , 0.9       ])

In [ ]:
#Oof! 100% recall for class 0 so over predicting class 0,  model overall is best at predicting class 2. 

recall_score(
    y_true=wine_test["class"],
    y_pred=wine_test["predicted"],
    average=None  
)

array([1.        , 0.38888889, 0.75      ])

In [154]:
knn = KNeighborsClassifier(n_neighbors=10)
X = wine_train[["total_phenols", "malic_acid"]]
y = wine_train["class"]

returned_dictionary = cross_validate(
    estimator=knn,
    cv=10,    # setting up the cross validation number
    X=X,
    y=y
)

cv_10_df = pd.DataFrame(returned_dictionary)    # Converting it to pandas DataFrame

cv_10_df

,fit_time,score_time,test_score
0,0.001442,0.002519,0.785714
1,0.000982,0.002022,0.642857
2,0.000866,0.002142,0.714286
3,0.000959,0.001446,0.461538
4,0.000797,0.001585,0.769231
5,0.000782,0.001265,0.769231
6,0.000779,0.001305,0.846154
7,0.000725,0.001183,0.846154
8,0.001121,0.001638,0.846154
9,0.000761,0.001237,0.846154


In [155]:
cv_10_metrics = cv_10_df.agg(["mean", "sem"])

cv_10_metrics

,fit_time,score_time,test_score
mean,0.000921,0.001634,0.752747
sem,0.000070,0.000143,0.038714


In [156]:
parameter_grid = {
    "n_neighbors": range(1, 20),
}

In [159]:
wine_tune_grid = GridSearchCV(
    estimator=knn,
    param_grid=parameter_grid,
    cv=10
)

In [161]:
#This shows 6 is the best value for n_neighbors.

wine_tune_grid.fit(
    wine_train[["total_phenols", "malic_acid"]],
    wine_train["class"]
)

accuracies_grid = pd.DataFrame(wine_tune_grid.cv_results_)
accuracies_grid

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002315,0.001490,0.004502,0.003202,1,{'n_neighbors': 1},0.714286,0.571429,0.714286,0.615385,0.692308,0.923077,0.769231,0.923077,0.846154,0.615385,0.738462,0.119250,12
1,0.000707,0.000101,0.001118,0.000105,2,{'n_neighbors': 2},0.785714,0.571429,0.571429,0.615385,0.769231,0.846154,0.692308,0.923077,0.846154,0.769231,0.739011,0.115965,10
2,0.000654,0.000041,0.001074,0.000066,3,{'n_neighbors': 3},0.785714,0.571429,0.714286,0.615385,0.846154,0.846154,0.769231,0.846154,0.846154,0.769231,0.760989,0.094379,2
3,0.000666,0.000061,0.001104,0.000109,4,{'n_neighbors': 4},0.714286,0.571429,0.642857,0.692308,0.692308,0.846154,0.846154,0.769231,0.846154,0.769231,0.739011,0.088639,10
4,0.000642,0.000049,0.001070,0.000077,5,{'n_neighbors': 5},0.785714,0.642857,0.714286,0.615385,0.769231,0.846154,0.769231,0.769231,0.846154,0.769231,0.752747,0.072144,3
5,0.000624,0.000029,0.001007,0.000033,6,{'n_neighbors': 6},0.785714,0.714286,0.642857,0.692308,0.769231,0.769231,0.846154,0.769231,0.846154,0.846154,0.768132,0.065511,1
6,0.000607,0.000022,0.001029,0.000039,7,{'n_neighbors': 7},0.785714,0.714286,0.714286,0.615385,0.769231,0.769231,0.692308,0.769231,0.846154,0.769231,0.744505,0.059749,9
7,0.000622,0.000028,0.001042,0.000064,8,{'n_neighbors': 8},0.785714,0.714286,0.714286,0.538462,0.692308,0.769231,0.846154,0.692308,0.769231,0.846154,0.736813,0.085278,16
8,0.000607,0.000019,0.001018,0.000032,9,{'n_neighbors': 9},0.785714,0.642857,0.714286,0.461538,0.769231,0.769231,0.846154,0.846154,0.846154,0.846154,0.752747,0.116141,3
9,0.000620,0.000029,0.001023,0.000043,10,{'n_neighbors': 10},0.785714,0.642857,0.714286,0.461538,0.769231,0.769231,0.846154,0.846154,0.846154,0.846154,0.752747,0.116141,3


In [163]:
wine_tune_grid.best_params_

{'n_neighbors': 6}

In [169]:
# Initialize the model with the best n_neighbors value
best_knn = KNeighborsClassifier(n_neighbors=6)

# Fit the model on the training data
best_knn.fit(
    wine_train[["total_phenols", "malic_acid"]], 
    wine_train["class"]
)

# Make predictions on test data
test_predictions = best_knn.predict(wine_test[["total_phenols", "malic_acid"]])

# Calculate accuracy score
test_accuracy = accuracy_score(
    wine_test["class"], 
    test_predictions
)

print(f"Wine_Test_Accuracy: {test_accuracy}")



Wine_Test_Accuracy: 0.7111111111111111


# Criteria


| **Criteria**                                           | **Complete**                                      | **Incomplete**                                    |
|--------------------------------------------------------|---------------------------------------------------|--------------------------------------------------|
| **Data Inspection**                                    | Data is inspected for number of variables, observations and data types. | Data inspection is missing or incomplete.         |
| **Data Scaling**                                       | Data scaling or normalization is applied where necessary (e.g., using `StandardScaler`). | Data scaling or normalization is missing or incorrectly applied. |
| **Model Initialization**                               | The KNN model is correctly initialized and a random seed is set for reproducibility.            | The KNN model is not initialized, is incorrect, or lacks a random seed for reproducibility. |
| **Parameter Grid for `n_neighbors`**                   | The parameter grid for `n_neighbors` is correctly defined. | The parameter grid is missing or incorrectly defined. |
| **Cross-Validation Setup**                             | Cross-validation is set up correctly with 10 folds. | Cross-validation is missing or incorrectly set up. |
| **Best Hyperparameter (`n_neighbors`) Selection**       | The best value for `n_neighbors` is identified using the grid search results. | The best `n_neighbors` is not selected or incorrect. |
| **Model Evaluation on Test Data**                      | The model is evaluated on the test data using accuracy. | The model evaluation is missing or uses the wrong metric. |


## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Note:

If you like, you may collaborate with others in the cohort. If you choose to do so, please indicate with whom you have worked with in your pull request by tagging their GitHub username. Separate submissions are required.

### Submission Parameters:
* Submission Due Date: `11:59 PM - 05/18/2025`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_3.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/LCR/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-6-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
